# 🧼 Data Cleaning Lab: Survey Edition

Welcome to the data cleaning battlefield.

This notebook cleans a messy file called `survey_dirty.csv` using pandas.  
It is organized step by step so you can run each part, inspect the results, and document your cleaning choices.

## Goals

- Load and inspect the dataset
- Clean string fields
- Handle missing values
- Standardize categorical data
- Fix data types
- Validate emails
- Convert consent to Boolean
- Check duplicates
- Save cleaned data


## 1. Load and inspect the data

In [8]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("survey_dirty.csv")
df.head()

,id,name,email,gender,exercise_freq,age,height_cm,weight_kg,satisfaction,consent_given,submitted_at,favorite_color,notes
0,1,Alice Johnson,alice.johnson@example.com,F,Daily,25,165cm,60 kg,5,Yes,2024-03-01 09:15,Blue,NaN
1,2,Bob Smith,bob.smith@example,male,3-5x/week,thirty,180,82kg,4,Y,03/02/2024 10:30,green,likes running
2,3,Carla Gomez,carla.gomez@example.com,Female,sometimes,29,170 cm,,3,true,2024/03/03,red,
3,4,David Lee,david.lee@example.com,M,Never,NaN,175cm,75kg,NaN,No,March 4 2024,black,prefers email
4,5,NaN,eva.wang@example.com,female,1-2 times/week,22,160,55,4,yes,2024-03-05T14:20:00,purple,new member


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              27 non-null     int64 
 1   name            25 non-null     object
 2   email           27 non-null     object
 3   gender          27 non-null     object
 4   exercise_freq   26 non-null     object
 5   age             24 non-null     object
 6   height_cm       27 non-null     object
 7   weight_kg       27 non-null     object
 8   satisfaction    25 non-null     object
 9   consent_given   27 non-null     object
 10  submitted_at    26 non-null     object
 11  favorite_color  24 non-null     object
 12  notes           13 non-null     object
dtypes: int64(1), object(12)
memory usage: 2.9+ KB


In [25]:
df.columns.tolist()

['id',
 'name',
 'email',
 'gender',
 'exercise_freq',
 'age',
 'height_cm',
 'weight_kg',
 'satisfaction',
 'consent_given',
 'submitted_at',
 'favorite_color',
 'notes',
 'email_valid']

## 2. Clean up string fields

We will:
- strip leading/trailing whitespace from object columns
- convert blank strings to missing values


In [11]:
object_cols = df.select_dtypes(include=["object"]).columns

for col in object_cols:
    df[col] = df[col].astype(str).str.strip()

blank_like = {"": np.nan, "nan": np.nan, "None": np.nan, "N/A": np.nan, "n/a": np.nan, "null": np.nan}
df = df.replace(blank_like)

df.head()

,id,name,email,gender,exercise_freq,age,height_cm,weight_kg,satisfaction,consent_given,submitted_at,favorite_color,notes
0,1,Alice Johnson,alice.johnson@example.com,F,Daily,25,165cm,60 kg,5,Yes,2024-03-01 09:15,Blue,NaN
1,2,Bob Smith,bob.smith@example,male,3-5x/week,thirty,180,82kg,4,Y,03/02/2024 10:30,green,likes running
2,3,Carla Gomez,carla.gomez@example.com,Female,sometimes,29,170 cm,NaN,3,true,2024/03/03,red,NaN
3,4,David Lee,david.lee@example.com,M,Never,NaN,175cm,75kg,NaN,No,March 4 2024,black,prefers email
4,5,NaN,eva.wang@example.com,female,1-2 times/week,22,160,55,4,yes,2024-03-05T14:20:00,purple,new member


## 3. Handle missing values

First inspect missing values, then decide whether to drop, fill, or flag them.


In [12]:
df.isnull().sum().sort_values(ascending=False)

notes             15
age                3
satisfaction       3
favorite_color     3
name               2
exercise_freq      2
weight_kg          2
submitted_at       2
height_cm          1
consent_given      1
id                 0
email              0
gender             0
dtype: int64

In [13]:
required_identity = [col for col in ["name", "email"] if col in df.columns]
if required_identity:
    df = df.dropna(subset=required_identity)

df.shape

(25, 13)

## 4. Standardize categorical data

We will standardize:
- `gender` → Male / Female / Other
- `exercise_freq` → Never / Rarely / Sometimes / Often / Daily


In [14]:
if "gender" in df.columns:
    gender_map = {
        "m": "Male", "male": "Male", "man": "Male", "boy": "Male",
        "f": "Female", "female": "Female", "woman": "Female", "girl": "Female",
        "other": "Other", "nonbinary": "Other", "non-binary": "Other", "nb": "Other",
        "prefer not to say": "Other"
    }
    df["gender"] = (
        df["gender"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(gender_map)
        .fillna("Other")
    )

if "exercise_freq" in df.columns:
    exercise_map = {
        "never": "Never",
        "none": "Never",
        "0": "Never",
        "rarely": "Rarely",
        "seldom": "Rarely",
        "sometimes": "Sometimes",
        "1-2 times/week": "Sometimes",
        "1-2x/week": "Sometimes",
        "weekly": "Sometimes",
        "often": "Often",
        "3-5 times/week": "Often",
        "3-5x/week": "Often",
        "daily": "Daily",
        "every day": "Daily",
        "7x/week": "Daily"
    }
    df["exercise_freq"] = (
        df["exercise_freq"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(exercise_map)
        .fillna("Sometimes")
    )

df.head()

,id,name,email,gender,exercise_freq,age,height_cm,weight_kg,satisfaction,consent_given,submitted_at,favorite_color,notes
0,1,Alice Johnson,alice.johnson@example.com,Female,Daily,25,165cm,60 kg,5,Yes,2024-03-01 09:15,Blue,NaN
1,2,Bob Smith,bob.smith@example,Male,Often,thirty,180,82kg,4,Y,03/02/2024 10:30,green,likes running
2,3,Carla Gomez,carla.gomez@example.com,Female,Sometimes,29,170 cm,NaN,3,true,2024/03/03,red,NaN
3,4,David Lee,david.lee@example.com,Male,Never,NaN,175cm,75kg,NaN,No,March 4 2024,black,prefers email
5,6,Frank Hall,frank.hall@sample.org,Male,Sometimes,41,182cm,90kg,2,1,2024-03-06,orange,NaN


## 5. Fix data types

We will:
- convert `age`, `height_cm`, `weight_kg`, and `satisfaction` to numeric
- extract numbers from strings like `"175cm"` or `"72 kg"`
- convert `submitted_at` to datetime


In [15]:
def extract_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.extract(r'([-+]?\d*\.?\d+)')[0],
        errors="coerce"
    )

for col in ["age", "height_cm", "weight_kg", "satisfaction"]:
    if col in df.columns:
        df[col] = extract_numeric(df[col])

if "submitted_at" in df.columns:
    df["submitted_at"] = pd.to_datetime(df["submitted_at"], errors="coerce")

df.dtypes

id                         int64
name                      object
email                     object
gender                    object
exercise_freq             object
age                      float64
height_cm                float64
weight_kg                float64
satisfaction             float64
consent_given             object
submitted_at      datetime64[ns]
favorite_color            object
notes                     object
dtype: object

## 6. Validate and clean emails

We will:
- lowercase and strip emails
- flag invalid emails using a regex
- keep a helper column called `email_valid`
- remove invalid emails


In [16]:
if "email" in df.columns:
    df["email"] = df["email"].astype(str).str.strip().str.lower()
    email_pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
    df["email_valid"] = df["email"].str.match(email_pattern, na=False)

if "email" in df.columns:
    display(df[["email", "email_valid"]].head())
else:
    print("No email column found.")

,email,email_valid
0,alice.johnson@example.com,True
1,bob.smith@example,False
2,carla.gomez@example.com,True
3,david.lee@example.com,True
5,frank.hall@sample.org,True


In [17]:
if "email_valid" in df.columns:
    df = df[df["email_valid"]]

df.shape

(23, 14)

## 7. Convert consent to Boolean

We will standardize `consent_given` into True/False values.


In [18]:
if "consent_given" in df.columns:
    consent_map = {
        "yes": True, "y": True, "true": True, "1": True,
        "no": False, "n": False, "false": False, "0": False
    }
    df["consent_given"] = (
        df["consent_given"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(consent_map)
    )

if "consent_given" in df.columns:
    print(df["consent_given"].value_counts(dropna=False))
else:
    print("No consent_given column found.")

consent_given
True     16
False     7
Name: count, dtype: int64


## 8. Check for duplicates

We will:
- inspect full-row duplicates
- inspect duplicate IDs if an ID column exists
- drop duplicates


In [19]:
duplicate_rows = df.duplicated().sum()
print("Duplicate full rows:", duplicate_rows)

possible_id_cols = [col for col in ["id", "survey_id", "respondent_id", "user_id"] if col in df.columns]
if possible_id_cols:
    id_col = possible_id_cols[0]
    print(f"Duplicate values in {id_col}:", df[id_col].duplicated().sum())
else:
    print("No ID column found to check duplicate IDs.")

df = df.drop_duplicates()

df.shape

Duplicate full rows: 0
Duplicate values in id: 1


(23, 14)

## 9. Final missing-value cleanup

Now that types are fixed, fill remaining numeric or text values in a simple way.


In [20]:
numeric_cols = df.select_dtypes(include=["number"]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

object_cols = df.select_dtypes(include=["object"]).columns
for col in object_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna("Unknown")

if "consent_given" in df.columns and df["consent_given"].isnull().any():
    df["consent_given"] = df["consent_given"].fillna(False)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23 entries, 0 to 26
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              23 non-null     int64         
 1   name            23 non-null     object        
 2   email           23 non-null     object        
 3   gender          23 non-null     object        
 4   exercise_freq   23 non-null     object        
 5   age             23 non-null     float64       
 6   height_cm       23 non-null     float64       
 7   weight_kg       23 non-null     float64       
 8   satisfaction    23 non-null     float64       
 9   consent_given   23 non-null     bool          
 10  submitted_at    7 non-null      datetime64[ns]
 11  favorite_color  23 non-null     object        
 12  notes           23 non-null     object        
 13  email_valid     23 non-null     bool          
dtypes: bool(2), datetime64[ns](1), float64(4), int64(1), object(6)
me

## 10. Cleaned data review

In [21]:
df.head()

,id,name,email,gender,exercise_freq,age,height_cm,weight_kg,satisfaction,consent_given,submitted_at,favorite_color,notes,email_valid
0,1,Alice Johnson,alice.johnson@example.com,Female,Daily,25.0,165.0,60.0,5.0,True,2024-03-01 09:15:00,Blue,Unknown,True
2,3,Carla Gomez,carla.gomez@example.com,Female,Sometimes,29.0,170.0,66.0,3.0,True,NaT,red,Unknown,True
3,4,David Lee,david.lee@example.com,Male,Never,27.0,175.0,75.0,4.0,False,NaT,black,prefers email,True
5,6,Frank Hall,frank.hall@sample.org,Male,Sometimes,41.0,182.0,90.0,2.0,True,NaT,orange,Unknown,True
6,7,Grace Kim,grace.kim@example.com,Female,Daily,27.0,168.0,58.0,4.0,True,NaT,yellow,bad date,True


In [22]:
df.describe(include="all")

,id,name,email,gender,exercise_freq,age,height_cm,weight_kg,satisfaction,consent_given,submitted_at,favorite_color,notes,email_valid
count,23.000000,23,23,23,23,23.000000,23.000000,23.000000,23.000000,23,7,23,23,23
unique,NaN,22,22,3,5,NaN,NaN,NaN,NaN,2,NaN,20,9,1
top,NaN,Tina Perez,tina.perez@example.com,Female,Sometimes,NaN,NaN,NaN,NaN,True,NaN,black,Unknown,True
freq,NaN,2,2,10,8,NaN,NaN,NaN,NaN,16,NaN,2,15,23
mean,14.521739,NaN,NaN,NaN,NaN,30.521739,171.347826,68.608696,3.565217,NaN,2024-03-15 01:10:51.428571392,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,17.000000,158.000000,52.000000,1.000000,NaN,2024-03-01 09:15:00,NaN,NaN,NaN
25%,9.000000,NaN,NaN,NaN,NaN,23.500000,166.500000,59.500000,3.000000,NaN,2024-03-10 19:45:00,NaN,NaN,NaN
50%,15.000000,NaN,NaN,NaN,NaN,27.000000,170.000000,66.000000,4.000000,NaN,2024-03-17 12:00:00,NaN,NaN,NaN
75%,20.000000,NaN,NaN,NaN,NaN,36.000000,174.500000,76.000000,4.000000,NaN,2024-03-20 16:10:00,NaN,NaN,NaN
max,26.000000,NaN,NaN,NaN,NaN,60.000000,190.000000,92.000000,5.000000,NaN,2024-03-23 11:11:00,NaN,NaN,NaN


In [23]:
df.isnull().sum()

id                 0
name               0
email              0
gender             0
exercise_freq      0
age                0
height_cm          0
weight_kg          0
satisfaction       0
consent_given      0
submitted_at      16
favorite_color     0
notes              0
email_valid        0
dtype: int64

## 11. Save your cleaned data

In [24]:
df.to_csv("survey_clean.csv", index=False)
print("Saved cleaned file as survey_clean.csv")

Saved cleaned file as survey_clean.csv


## Notes on cleaning choices

Example justifications you can reuse:

- Rows missing key identity fields like `name` or `email` were dropped because they are harder to use reliably.
- Numeric fields were coerced to numbers so broken strings became `NaN`.
- Remaining numeric missing values were filled with the median to reduce distortion from outliers.
- Categorical fields were standardized to consistent labels for easier analysis.
- Invalid emails were filtered out because they fail basic format validation.
